# Lab 04D: The Research Pipeline

**Week 4 &mdash; Agentic AI: Building Autonomous Intelligent Systems** &middot; *Graded assignment*

This is the graded assignment for Week 4, and it is a **composition** exercise: you connect the pieces you learned in Labs 04A and 04B into one working pipeline. A crew **plans** a research job, **reads three live Wikipedia pages** with a prebuilt scrape tool, and writes a five-sentence brief &mdash; then a **Flow state machine** puts that brief through a quality gate and routes it to publish or repair.

```
                      THE PIPELINE YOU ARE BUILDING

 topic: "AI agents"
     |
     v
 +-----------------------------------------------------------------+
 |  PART 1 -- the research crew          (planning=True, Lab 04B)  |
 |                                                                 |
 |   researcher --- ScrapeWebsiteTool (Lab 04A) ---> reads three   |
 |       |                                          Wikipedia      |
 |       |  context handoff (Lab 04B)               pages          |
 |       v                                                         |
 |   writer -----> five-sentence brief                             |
 +-----------------------------------------------------------------+
     |
     |  the brief (a string)
     v
 +-----------------------------------------------------------------+
 |  PART 2 -- the quality gate      (a Flow state machine, 04B)    |
 |                                                                 |
 |   @start   measure : count the brief's words                    |
 |      |                                                          |
 |   @router  gate    : within the word budget?                    |
 |      |--> "PASS" --> publish : write research_brief.md          |
 |      |--> "FAIL" --> revise  : LLM shortens it, then write      |
 +-----------------------------------------------------------------+
     |
     v
 research_brief.md
```

## How this assignment works

This notebook gives you **specifications and diagrams &mdash; not example code to copy.** Every pattern you need was taught in this week's walkthrough labs; open them side by side and work from them:

| Piece you build | Where it was taught |
|---|---|
| A scrape tool the agent steers | Lab 04A &mdash; "A prebuilt tool: read a real web page" |
| Agents, tasks, the `context` handoff | Lab 04B research crew (and Week 3) |
| A crew that plans before executing | Lab 04B &mdash; plan-first execution |
| A Flow with `@start` / `@router` / `@listen` | Lab 04B &mdash; "Control structure" state machine |

## Grading (10 pts)

| Piece | Points |
|---|---|
| TODO 1 &mdash; the scrape tool | 1 |
| TODO 2 &mdash; the two agents | 2 |
| TODO 3 &mdash; the two tasks + handoff | 2 |
| TODO 4 &mdash; the planning crew, kicked off | 2 |
| TODO 5 &mdash; the quality-gate Flow | 2 |
| Clean top-to-bottom run that produces `research_brief.md` | 1 |

## Setup

Add your key as a Colab secret: get one from [Google AI Studio](https://aistudio.google.com/app/apikey), click the **key icon** in Colab, add a secret named **`GEMINI_API_KEY`**, and enable **Notebook access**. The next cell installs CrewAI and its prebuilt-tools package (give it a minute), and the one after builds the LLM.

In [1]:
!pip install -q crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

> **Heads-up on the pip output:** an `ERROR: pip's dependency resolver ...` line about `bigframes`/`rich` is **expected in Colab and safe to ignore**. If Colab offers a **"Restart session"** prompt, click it and re-run from setup.

In [2]:
import os

from google.colab import userdata
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")
os.environ["GEMINI_API_KEY"] = api_key

# gemini-2.5-flash (not flash-lite): a tool-using crew fires many multi-step requests, and the
# lite model is often overloaded (HTTP 503) under that load. num_retries adds a per-call cushion.
llm = LLM(model="gemini/gemini-3.1-flash-lite", api_key=api_key, temperature=0.3, num_retries=5)

# Keep output clean: quiet CrewAI/LiteLLM ERROR logs, hide a legacy-SDK notice, no run traces.
import logging, warnings
for _n in ("crewai.flow.runtime", "LiteLLM", "litellm", "root"):
    logging.getLogger(_n).setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message=r".*google\.generativeai.*")
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Silence CrewAI's tracing notices (the repeating "Tracing Preference" panels) so only the clean
# execution boxes show. The first-time flag is captured when crewai is imported, so we also clear
# it directly on the listener. (Internal API -- guarded.)
try:
    from crewai.events.listeners.tracing.utils import (
        set_suppress_tracing_messages, mark_first_execution_done,
    )
    set_suppress_tracing_messages(True)
    mark_first_execution_done()
    from crewai.events.listeners.tracing.trace_listener import TraceCollectionListener
    if TraceCollectionListener._instance is not None:
        TraceCollectionListener._instance.first_time_handler.is_first_time = False
except Exception:
    pass

print("Setup ready.")

Setup ready.


## Part 1 &mdash; The research crew

Two agents, two tasks, one handoff &mdash; and the crew **plans first** because this is exactly the kind of job planning earns its cost on: multiple pages to visit, in a predictable order, feeding one synthesis.

```
   research_task                              brief_task
   agent: researcher (has the scrape tool)    agent: writer (no tools)
        |                                          ^
        |   facts from all three pages             |
        +------------------------------------------+
                     context handoff
```

### TODO 1 &mdash; the scrape tool

In Lab 04A you built `ScrapeWebsiteTool` **pinned to one page** with `website_url=...`. Your researcher must read **three different pages with the same tool**, so pin nothing &mdash; construct it so the agent supplies the URL itself on each call. Lab 04A told you this variant exists; the code comment in that lab says exactly when to use it.

In [3]:
from crewai_tools import ScrapeWebsiteTool

# ------------------------- YOUR CODE HERE (TODO 1) -------------------------
# Create the researcher's scrape tool in a variable named `scrape`.
# Requirement: the AGENT chooses which URL to read on each call -- do not pin
# the tool to a single page. (Lab 04A, "A prebuilt tool: read a real web page".)
scrape = ScrapeWebsiteTool()

#raise NotImplementedError("Replace this line with your scrape tool (TODO 1)")

### TODO 2 &mdash; the two agents

Same `Agent` shape as every crew this week: role, goal, backstory, `llm`. What matters here is **who gets which tools**:

- **`researcher`** &mdash; a web researcher. Give it the scrape tool from TODO 1. Its backstory should insist that it reads every assigned page and never invents facts.
- **`writer`** &mdash; a technical writer that turns research notes into tight briefs. Give it **no tools**. That is least privilege from the Module A lecture: the writer's job never touches the web, so it gets no way to.

In [4]:
# ------------------------- YOUR CODE HERE (TODO 2) -------------------------
# Build the two agents described above: `researcher` (with the scrape tool)
# and `writer` (no tools). Both use the `llm` from setup.
#raise NotImplementedError("Replace this line with your two agents (TODO 2)")

researcher = Agent(role="Web Researcher", goal="Gather concrete facts on the topic.",
                   backstory="You should read every assigned page and never invents facts.",
                   tools=[scrape], llm=llm)
writer = Agent(role="Technical Writer", goal="Write a brief summary.",
               backstory="You write clear prose for a general audience.", llm=llm)

### TODO 3 &mdash; the two tasks, wired together

The researcher's output must flow into the writer's context &mdash; the same task-to-task handoff (`context=[...]`) you used in Lab 04B.

- **`research_task`** (agent: `researcher`) &mdash; the description must tell the agent to read **each** of these three Wikipedia pages with its scrape tool and extract 2&ndash;3 key facts from each:
  - `https://en.wikipedia.org/wiki/Intelligent_agent`
  - `https://en.wikipedia.org/wiki/Multi-agent_system`
  - `https://en.wikipedia.org/wiki/Software_agent`

  `expected_output`: a bullet list of 6&ndash;9 facts, grouped under the page each came from.
- **`brief_task`** (agent: `writer`) &mdash; using **only the researcher's facts**, write a brief on AI agents: what they are, how multi-agent systems cooperate, and where software agents show up. **Five sentences maximum.** `expected_output`: a brief of at most five sentences. Wire the handoff so the writer actually receives the researcher's facts.

Give both tasks a `name=` &mdash; it is what makes the verbose boxes readable.

In [5]:
# ------------------------- YOUR CODE HERE (TODO 3) -------------------------
# Build `research_task` and `brief_task` to the specs above, and hand the
# researcher's output to the writer the way Lab 04B wired its tasks.
#raise NotImplementedError("Replace this line with your two tasks (TODO 3)")

research_task = Task(name="research",
                     description="Read each of the 3 Wikipedia pages - https://en.wikipedia.org/wiki/Intelligent_agent, https://en.wikipedia.org/wiki/Multi-agent_system, https://en.wikipedia.org/wiki/Software_agent",
                     expected_output="Extract 3 key facts from each page.",
                     agent=researcher)
brief_task = Task(name="brief",
                  description="Write a brief from the researcher's facts on AI agents: what they are, how multi-agent systems cooperate, and where software agents show up",
                  expected_output="A brief of at most five sentences.",
                  agent=writer,
                  context=[research_task])

### TODO 4 &mdash; assemble the crew, plan, and run

Put it together the way Lab 04B did: both agents, both tasks, `Process.sequential`, and &mdash; because this job has multiple predictable steps &mdash; make the crew **draft a plan before executing**. Remember from 04B that turning planning on takes *two* arguments, and that in Colab a crew is kicked off with the **async** call.

Store the crew's final output **as a string in a variable named `brief`** &mdash; Part 2 reads that exact name.

In [6]:
# ------------------------- YOUR CODE HERE (TODO 4) -------------------------
# Assemble the crew (both agents, both tasks, sequential), enable planning,
# kick it off with the async call, and store the final output as the string `brief`.
#raise NotImplementedError("Replace this line with your crew (TODO 4)")

planning_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, brief_task],
    process=Process.sequential,
    planning=True,           # <-- draft a plan before executing
    planning_llm=llm,
    # verbose stays OFF here: with planning=True, a verbose run also prints CrewAI's internal
    # planner box (a raw metadata dump). We already printed the plan cleanly above, so this cell
    # just executes and prints the result. The clean step-by-step boxes appear in the runs below.
)
plan_result = await planning_crew.kickoff_async()
print("\nFinal result:\n", plan_result)
brief = str(plan_result)
print("CrewAI completed. Generating flow pipeline...")


Final result:
 Intelligent agents are autonomous entities that perceive their environment and take goal-oriented actions to achieve optimal outcomes. When multiple agents collaborate in a multi-agent system, they utilize structured protocols like negotiation and consensus-building to solve complex problems through decentralized coordination. These systems benefit from self-organization and fault tolerance, as no single agent possesses a complete global view. Software agents implement these principles by acting persistently on behalf of users or other programs without constant human intervention. Today, they are widely deployed in diverse fields such as e-commerce shopping bots, system monitoring, data mining, and cybersecurity.
CrewAI completed. Generating flow pipeline...


## Part 2 &mdash; The quality gate (a Flow state machine)

A crew *produces*; a Flow *governs*. Your brief now goes through an explicit state machine &mdash; the same `@start` / `@router` / `@listen` pattern as Lab 04B's `ModerationFlow`, with two differences that make it yours: the **router decides in plain Python** (no LLM needed to count words), and the **FAIL branch does real repair work** instead of just labeling.

```
 kickoff(inputs={"brief": brief})
     |
     v
 @start    measure : word_count = number of words in the brief
     |
     v
 @router   gate    : return "PASS" if word_count <= WORD_BUDGET else "FAIL"
     |
     |--> @listen("PASS")  publish : write the brief to research_brief.md
     |
     |--> @listen("FAIL")  revise  : llm.call(...) shortens the brief to the
                                     budget, update the state, then write
                                     research_brief.md
```

**Either path is a correct run.** The gate's job is to make the *outcome* uniform: whatever the writer produced, what lands in `research_brief.md` respects the budget. Note the state model is given &mdash; your flow reads and writes its fields; set `action` in both terminal steps so the run is auditable.

### TODO 5 &mdash; build `BriefGateFlow`

Requirements, matching the diagram exactly:

- Class **`BriefGateFlow(Flow[BriefState])`** &mdash; the run cell below constructs it by this name.
- The `@start()` step computes `self.state.word_count` from `self.state.brief` (plain Python).
- The `@router(...)` step returns the **label** `"PASS"` or `"FAIL"` &mdash; routers return labels, they do not act.
- `@listen("PASS")` writes `self.state.brief` to `research_brief.md` and sets `self.state.action`.
- `@listen("FAIL")` uses `llm.call(...)` to shorten the brief to at most `WORD_BUDGET` words, updates `self.state.brief` and `self.state.word_count`, writes `research_brief.md`, and sets `self.state.action`.
- Have each step `print` what it did, so the path through the machine is visible in the output.

In [7]:
from pydantic import BaseModel
from crewai.flow.flow import Flow, listen, start, router

WORD_BUDGET = 120

# The typed state your flow carries -- given. Your steps read and write these fields.
class BriefState(BaseModel):
    brief: str = ""
    word_count: int = 0
    action: str = ""

class BriefGateFlow(Flow[BriefState]):

    @start()
    def count_words(self):
        words = self.state.brief.split()
        self.state.word_count = len(words)
        print(f"Flow started. Brief word count: {self.state.word_count}")

    @router(count_words)
    def route_budget(self):
        if self.state.word_count <= WORD_BUDGET:
            return "PASS"
        else:
            return "FAIL"

    @listen("PASS")
    def accept_brief(self):
        self.state.action = "Approved and Saved"
        with open("research_brief.md", "w") as f:
            f.write(self.state.brief)
        print("-> Brief passed budget criteria. Saved to research_brief.md")

    @listen("FAIL")
    def reject_brief(self):
        self.state.action = "Rejected: Exceeds Budget"
        error_message = f"Budget Error: Brief was {self.state.word_count} words (Limit: {WORD_BUDGET}).\n\n{self.state.brief}"
        with open("research_brief.md", "w") as f:
            f.write(error_message)
        print(f"-> Brief failed budget criteria ({self.state.word_count} words).")
# ------------------------- YOUR CODE HERE (TODO 5) -------------------------
# Build BriefGateFlow(Flow[BriefState]) to the spec above: one @start step,
# one @router, and the "PASS" / "FAIL" @listen branches from the diagram.
#raise NotImplementedError("Replace this line with your BriefGateFlow class (TODO 5)")

## Run the pipeline end to end

This cell is given. It pushes your crew's brief through your gate and prints the artifact. A successful run shows the path the machine took (`measure` &rarr; `gate` &rarr; `publish` *or* `revise`) followed by the contents of `research_brief.md`.

In [8]:

flow = BriefGateFlow()
flow.kickoff(inputs={"brief": brief})

print("\naction taken:", flow.state.action)
print("\n--- research_brief.md ---\n")
print(open("research_brief.md").read())

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: BriefGateFlow                                                                                            │
│  ID: cb243858-1d8d-4ce7-96b1-1f4a5a7767f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: BriefGateFlow                                                                                            │
│  ID: cb243858-1d8d-4ce7-96b1-1f4a5a7767f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started. Brief word count: 95


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: count_words                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: route_budget                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: accept_brief                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: route_budget                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: accept_brief                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: BriefGateFlow                                                                                            │
│  ID: cb243858-1d8d-4ce7-96b1-1f4a5a7767f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


action taken: Approved and Saved

--- research_brief.md ---

Intelligent agents are autonomous entities that perceive their environment and take goal-oriented actions to achieve optimal outcomes. When multiple agents collaborate in a multi-agent system, they utilize structured protocols like negotiation and consensus-building to solve complex problems through decentralized coordination. These systems benefit from self-organization and fault tolerance, as no single agent possesses a complete global view. Software agents implement these principles by acting persistently on behalf of users or other programs without constant human intervention. Today, they are widely deployed in diverse fields such as e-commerce shopping bots, system monitoring, data mining, and cybersecurity.


## Submit

1. Run the notebook **top to bottom** with all five TODOs completed &mdash; the last cell must show the gate's path and the final `research_brief.md`.
2. Push your notebook to your course GitHub repo (or share the Colab link with view access).
3. Paste the URL into the Canvas assignment.

**Checklist before you submit**

- The notebook runs top to bottom without errors.
- The crew output shows the researcher reading the three Wikipedia pages and the writer's five-sentence brief.
- The flow output shows `measure`, the gate's decision, and the terminal step &mdash; and `research_brief.md` is printed at the end.
- Your API key is **not** pasted into any code cell.

### Recap &mdash; what you composed

- A **prebuilt tool** the agent steers (Lab 04A) gave the crew real-world reach: three live pages, one tool.
- **`planning=True`** (Lab 04B) made the crew draft the multi-page strategy before the first scrape.
- The **`context` handoff** carried the researcher's facts into the writer's task.
- A **Flow state machine** (Lab 04B) governed the result: a plain-Python **router** chose the path, and the FAIL branch *repaired* instead of rejecting.

That is the Week 4 story in one pipeline: tools give agents hands, planning gives them a strategy, and explicit control structure decides what may happen next.